In [1]:
%%configure -f
{
    "defaultlakehouse":{
        "name":"ACME_LH"
    }
}

StatementMeta(, 123b1a06-b6a9-4a7d-9cd2-e22f31d3d9fa, -1, Finished, Available, Finished, True)

In [3]:
%%sql
SELECT * FROM ACME_LH.dbo.supplier;
SELECT * FROM ACME_LH.dbo.budget;
SELECT * FROM ACME_LH.dbo.actual;

StatementMeta(, 123b1a06-b6a9-4a7d-9cd2-e22f31d3d9fa, 6, Finished, Available, Finished, True)

<Spark SQL result set with 12 rows and 11 fields>

<Spark SQL result set with 29 rows and 13 fields>

<Spark SQL result set with 214 rows and 13 fields>

In [2]:
%%sql
DROP TABLE IF EXISTS ACME_LH.dbo.actual;

CREATE TABLE ACME_LH.dbo.actual
(
    Period          INT,
    InvoiceNumber   STRING,
    InvoiceDate     date,
    PurchaseOrderNo STRING,
    CurrencyCode    STRING,
    AmountLocal     DOUBLE,
    Description     STRING,
    version         INT,
    source_file     STRING,
    loaded_at       TIMESTAMP 
)
USING DELTA;

StatementMeta(, 6c261ae3-f626-4871-82aa-256733383569, 3, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
%%sql
DROP TABLE IF EXISTS ACME_LH.dbo.budget;

CREATE TABLE ACME_LH.dbo.budget
(
    suppliercode       STRING,
    period             INT,
    currencycode       STRING,
    budgetamountlocal  DOUBLE,
    wbscode  STRING,
    costtype STRING,
    budgetnote         STRING,
    budget_year        INT,
    budget_month       INT,
    budget_period      DATE,
    version            INT,
    source_file        STRING,
    loaded_at          TIMESTAMP
)
USING DELTA;

StatementMeta(, c60596e8-c016-41ea-98c0-f4dc0ad2dbab, 3, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
%%sql
DROP TABLE IF EXISTS ACME_LH.dbo.supplier;

CREATE TABLE ACME_LH.dbo.supplier
(
    SupplierCode      STRING,
    SupplierName      STRING,
    CountryName       STRING,
    CurrencyCode      STRING,
    SpendCategory     STRING,
    PaymentTermsDays  INT,
    version           INT,
    source_file       STRING,
    loaded_at         TIMESTAMP 
)
USING DELTA;

StatementMeta(, b63899d1-8d15-4cb3-8ac8-1cfbf79c147a, 3, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
%%sql
TRUNCATE TABLE dbo.budget;
TRUNCATE TABLE dbo.actual;
TRUNCATE TABLE dbo.supplier;

StatementMeta(, 20e77bd7-04f5-4c17-90ec-3049299b92f6, 9, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
%%sql
TRUNCATE TABLE dbo.Budget_Load_log

In [1]:
%%sql
SELECT * From dbo.budget

StatementMeta(, d14133ff-3321-4eb8-949c-d3fa6b978812, 2, Finished, Available, Finished, True)

Error: No default context found, please attach a lakehouse before running spark sql queries with partial namespaces.

In [ ]:
%%sql
SELECT * From dbo.Budget_Load_log

In [2]:
# =============================================================
# Budget file loader – Microsoft Fabric Lakehouse
# File naming convention: yyyy_mm_budget_vNN.csv  e.g. 2026_04_budget_v01.csv
# Loads Files/budgets/landing/*.csv into Delta table 'budget'
# then moves each file to archive/ (success) or error/ (failure)
# =============================================================
import re
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F

# ---------- Configuration ----------
BASE_PATH = "Files/budgets"
LANDING_PATH = f"{BASE_PATH}/landing"
ARCHIVE_PATH = f"{BASE_PATH}/archive"
ERROR_PATH  = f"{BASE_PATH}/error"
TARGET_TABLE = "budget"

# yyyy_mm_budget_vNN.csv
FILENAME_PATTERN = re.compile(
    r"^(?P<budget_year>20\d{2})_"
    r"(?P<budget_month>0[1-9]|1[0-2])_"
    r"budget_"
    r"v(?P<version>\d{1,2})"
    r"\.csv$",
    re.IGNORECASE,
)

run_ts = datetime.now(timezone.utc)
archive_folder = f"{ARCHIVE_PATH}/{run_ts:%Y/%m/%d}"

# ---------- Helpers ----------
def clean_column_name(name: str) -> str:
    """Delta-safe column names: lowercase, underscores, no special chars."""
    name = re.sub(r"[^0-9a-zA-Z]+", "_", str(name).strip()).strip("_").lower()
    return name or "unnamed"


def read_csv(path: str):
    df = (spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path))
    return df.toDF(*[clean_column_name(c) for c in df.columns])


def move_file(src: str, dest_folder: str):
    notebookutils.fs.mkdirs(dest_folder)
    notebookutils.fs.mv(src, f"{dest_folder}/{src.split('/')[-1]}", overwrite=True)


# ---------- Main ----------
notebookutils.fs.mkdirs(LANDING_PATH)
files = sorted(
    [f for f in notebookutils.fs.ls(LANDING_PATH) if not f.isDir],
    key=lambda f: f.name, # yyyy_mm_..._vNN sorts chronologically
)
print(f"Found {len(files)} file(s) in {LANDING_PATH}")

results = []

for f in files:
        file_name = f.name
        src_path = f"{LANDING_PATH}/{file_name}"
        try:
            # 1. Validate the file name
            m = FILENAME_PATTERN.match(file_name)
            if not m:
                raise ValueError("File name does not match yyyy_mm_budget_vNN.csv")
            
            year = int(m["budget_year"])
            month  = int(m["budget_month"])
            version = int(m["version"])

            # 2. Read and add metadata columns parsed from the file name
            df = read_csv(src_path)
            row_count = df.count()

            if row_count == 0:
                raise ValueError("File contains no rows")

            df = (df
                .withColumn("budget_year",  F.lit(year))
                .withColumn("budget_month", F.lit(month))
                .withColumn("budget_period", F.make_date(F.lit(year), F.lit(month), F.lit(1)))
                .withColumn("version", F.lit(version))
                .withColumn("source_file", F.lit(file_name))
                .withColumn("loaded_at", F.lit(run_ts)))

            # 3. Idempotent load: remove rows from a previous load of the same file
            if spark.catalog.tableExists(TARGET_TABLE):
                (DeltaTable.forName(spark, TARGET_TABLE)
                .delete(F.col("source_file") == file_name))

            (df.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(TARGET_TABLE))

            # 4. Move to archive only after a successful write
            move_file(src_path, archive_folder)
            results.append((file_name, "loaded", row_count, None))
            print(f"✔ {file_name}: {row_count} rows loaded")

        except Exception as e:
            move_file(src_path, ERROR_PATH)
            results.append((file_name, "error", 0, str(e)[:500]))
            print(f"✘ {file_name}: {e}")


# ---------- Load log ----------
if results:
    log_df = (spark.createDataFrame(
                results, "file_name string, status string, row_count long, error_message string")
                .withColumn("run_at", F.lit(run_ts)))
    log_df.write.format("delta").mode("append").saveAsTable("budget_load_log")
    display(log_df)

StatementMeta(, 123b1a06-b6a9-4a7d-9cd2-e22f31d3d9fa, 3, Finished, Available, Finished, False)

Found 1 file(s) in Files/budgets/landing
✔ 2026_04_budget_v1.csv: 29 rows loaded


SynapseWidget(Synapse.DataFrame, adcdf6cf-eddd-4035-a0c4-7d0d54f1b63d)

In [7]:
# =============================================================
# Suppliers file loader – Microsoft Fabric Lakehouse
# File naming convention: yyyy_mm_supplier_vNN.csv  e.g. 2026_04_supplier_v01.csv
# Loads Files/suppliers/landing/*.csv into Delta table 'supplier'
# then moves each file to archive/ (success) or error/ (failure)
# =============================================================
import re
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F

# ---------- Configuration ----------
BASE_PATH = "Files/suppliers"
LANDING_PATH = f"{BASE_PATH}/landing"
ARCHIVE_PATH = f"{BASE_PATH}/archive"
ERROR_PATH  = f"{BASE_PATH}/error"
TARGET_TABLE = "supplier"

# yyyy_mm_supplier_vNN.csv
FILENAME_PATTERN = re.compile(
    r"^(?P<file_year>20\d{2})_"
    r"(?P<file_month>0[1-9]|1[0-2])_"
    r"supplier_"
    r"v(?P<version>\d{1,2})"
    r"\.csv$",
    re.IGNORECASE,
)
run_ts = datetime.now(timezone.utc)
archive_folder = f"{ARCHIVE_PATH}/{run_ts:%Y/%m/%d}"

# ---------- Helpers ----------
def clean_column_name(name: str) -> str:
    """Delta-safe column names: lowercase, underscores, no special chars."""
    name = re.sub(r"[^0-9a-zA-Z]+", "_", str(name).strip()).strip("_").lower()
    return name or "unnamed"


def read_csv(path: str):
    df = (spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path))
    return df.toDF(*[clean_column_name(c) for c in df.columns])


def move_file(src: str, dest_folder: str):
    notebookutils.fs.mkdirs(dest_folder)
    notebookutils.fs.mv(src, f"{dest_folder}/{src.split('/')[-1]}", overwrite=True)


# ---------- Main ----------
notebookutils.fs.mkdirs(LANDING_PATH)
files = sorted(
    [f for f in notebookutils.fs.ls(LANDING_PATH) if not f.isDir],
    key=lambda f: f.name, # yyyy_mm_..._vNN sorts chronologically
)
print(f"Found {len(files)} file(s) in {LANDING_PATH}")

results = []

for f in files:
        file_name = f.name
        src_path = f"{LANDING_PATH}/{file_name}"
        try:
            # 1. Validate the file name
            m = FILENAME_PATTERN.match(file_name)
            if not m:
                raise ValueError("File name does not match yyyy_mm_supplier_vNN.csv")
            
            year = int(m["file_year"])
            month  = int(m["file_month"])
            version = int(m["version"])

            # 2. Read and add metadata columns parsed from the file name
            df = read_csv(src_path)
            row_count = df.count()

            if row_count == 0:
                raise ValueError("File contains no rows")

            df = (df
                .withColumn("file_year",  F.lit(year))
                .withColumn("file_month", F.lit(month))
                .withColumn("version", F.lit(version))
                .withColumn("source_file", F.lit(file_name))
                .withColumn("loaded_at", F.lit(run_ts)))

            # 3. Idempotent load: remove rows from a previous load of the same file
            if spark.catalog.tableExists(TARGET_TABLE):
                (DeltaTable.forName(spark, TARGET_TABLE)
                .delete(F.col("source_file") == file_name))

            (df.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(TARGET_TABLE))

            # 4. Move to archive only after a successful write
            move_file(src_path, archive_folder)
            results.append((file_name, "loaded", row_count, None))
            print(f"✔ {file_name}: {row_count} rows loaded")

        except Exception as e:
            move_file(src_path, ERROR_PATH)
            results.append((file_name, "error", 0, str(e)[:500]))
            print(f"✘ {file_name}: {e}")


# ---------- Load log ----------
if results:
    log_df = (spark.createDataFrame(
                results, "file_name string, status string, row_count long, error_message string")
                .withColumn("run_at", F.lit(run_ts)))
    log_df.write.format("delta").mode("append").saveAsTable("supplier_load_log")
    display(log_df)

StatementMeta(, 20e77bd7-04f5-4c17-90ec-3049299b92f6, 12, Finished, Available, Finished, False)

Found 1 file(s) in Files/suppliers/landing
✔ 2026_04_supplier_v1.csv: 12 rows loaded


SynapseWidget(Synapse.DataFrame, 309a6e9f-67f4-49e0-9c04-08c84cf8f78a)

In [8]:
# =============================================================
# Actual file loader – Microsoft Fabric Lakehouse
# File naming convention: yyyy_mm_actual_vNN.csv  e.g. 2026_04_actual_v01.csv
# Loads Files/actuals/landing/*.csv into Delta table 'actual'
# then moves each file to archive/ (success) or error/ (failure)
# =============================================================
import re
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F

# ---------- Configuration ----------
BASE_PATH = "Files/actuals"
LANDING_PATH = f"{BASE_PATH}/landing"
ARCHIVE_PATH = f"{BASE_PATH}/archive"
ERROR_PATH  = f"{BASE_PATH}/error"
TARGET_TABLE = "actual"

# yyyy_mm_actual_vNN.csv
FILENAME_PATTERN = re.compile(
    r"^(?P<actual_year>20\d{2})_"
    r"(?P<actual_month>0[1-9]|1[0-2])_"
    r"actual_"
    r"v(?P<version>\d{1,2})"
    r"\.csv$",
    re.IGNORECASE,
)

run_ts = datetime.now(timezone.utc)
archive_folder = f"{ARCHIVE_PATH}/{run_ts:%Y/%m/%d}"

# ---------- Helpers ----------
def clean_column_name(name: str) -> str:
    """Delta-safe column names: lowercase, underscores, no special chars."""
    name = re.sub(r"[^0-9a-zA-Z]+", "_", str(name).strip()).strip("_").lower()
    return name or "unnamed"


def read_csv(path: str):
    df = (spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path))
    return df.toDF(*[clean_column_name(c) for c in df.columns])


def move_file(src: str, dest_folder: str):
    notebookutils.fs.mkdirs(dest_folder)
    notebookutils.fs.mv(src, f"{dest_folder}/{src.split('/')[-1]}", overwrite=True)


# ---------- Main ----------
notebookutils.fs.mkdirs(LANDING_PATH)
files = sorted(
    [f for f in notebookutils.fs.ls(LANDING_PATH) if not f.isDir],
    key=lambda f: f.name, # yyyy_mm_..._vNN sorts chronologically
)
print(f"Found {len(files)} file(s) in {LANDING_PATH}")

results = []

for f in files:
        file_name = f.name
        src_path = f"{LANDING_PATH}/{file_name}"
        try:
            # 1. Validate the file name
            m = FILENAME_PATTERN.match(file_name)
            
            if not m:
                raise ValueError("File name does not match yyyy_mm_actual_vNN.csv")
            
            year = int(m["actual_year"])
            month  = int(m["actual_month"])
            version = int(m["version"])

            # 2. Read and add metadata columns parsed from the file name
            df = read_csv(src_path)
            row_count = df.count()

            if row_count == 0:
                raise ValueError("File contains no rows")

            df = (df
                .withColumn("actual_year",  F.lit(year))
                .withColumn("actual_month", F.lit(month))
                .withColumn("actual", F.make_date(F.lit(year), F.lit(month), F.lit(1)))
                .withColumn("version", F.lit(version))
                .withColumn("source_file", F.lit(file_name))
                .withColumn("loaded_at", F.lit(run_ts)))

            # 3. Idempotent load: remove rows from a previous load of the same file
            if spark.catalog.tableExists(TARGET_TABLE):
                (DeltaTable.forName(spark, TARGET_TABLE)
                .delete(F.col("source_file") == file_name))

            (df.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(TARGET_TABLE))

            # 4. Move to archive only after a successful write
            move_file(src_path, archive_folder)
            results.append((file_name, "loaded", row_count, None))
            print(f"✔ {file_name}: {row_count} rows loaded")

        except Exception as e:
            move_file(src_path, ERROR_PATH)
            results.append((file_name, "error", 0, str(e)[:500]))
            print(f"✘ {file_name}: {e}")


# ---------- Load log ----------
if results:
    log_df = (spark.createDataFrame(
                results, "file_name string, status string, row_count long, error_message string")
                .withColumn("run_at", F.lit(run_ts)))
    log_df.write.format("delta").mode("append").saveAsTable("actual_load_log")
    display(log_df)

StatementMeta(, 20e77bd7-04f5-4c17-90ec-3049299b92f6, 13, Finished, Available, Finished, False)

Found 9 file(s) in Files/actuals/landing
✔ 2026_04_actual_v1.csv: 24 rows loaded
✔ 2026_05_actual_v1.csv: 24 rows loaded
✔ 2026_06_actual_v1.csv: 24 rows loaded
✔ 2026_07_actual_v1.csv: 24 rows loaded
✔ 2026_08_actual_v1.csv: 24 rows loaded
✔ 2026_09_actual_v1.csv: 24 rows loaded
✔ 2026_10_actual_v1.csv: 24 rows loaded
✔ 2026_11_actual_v1.csv: 24 rows loaded
✔ 2026_12_actual_v1.csv: 22 rows loaded


SynapseWidget(Synapse.DataFrame, fbd1dbbb-8ef9-4e3b-b7e5-983fe8a8801f)